<a href="https://colab.research.google.com/github/amritaa72/Runs-and-Wickets/blob/main/WBBL_HTML_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [130]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
url= "https://cricketsix.com/match/162733/sydney-thunder-women-vs-brisbane-heat-women-6th-match-wbbl-2026/"
response= requests.get(url, timeout=20)
print("status code :",response.status_code)
soup= BeautifulSoup(response.text, "html.parser")
print("Page Title : ",soup.title.get_text(strip=True) if soup.title else "Not Found")

page_text = soup.get_text("\n", strip=True)

keyword = "Ball-by-ball commentary"
position = page_text.find(keyword)

if position != -1:
    print("\nCommentary heading found!")
    print(page_text[position:position + 2500])
else:
    print("\nCommentary heading NOT found in downloaded HTML.")


status code : 200
Page Title :  Sydney Thunder Women vs Brisbane Heat Women Scorecard & Commentary, 6th Match | WBBL 2026 | CricketSix.com

Commentary heading found!
Ball-by-ball commentary
All
Wickets
4s & 6s
Overs
16.4
4
Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!
Nadine de Klerk comes in, a right-handed bat.
16.3
W
Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harris!! What a stunning catch, and it's a big wicket too, Laura Harris is caught by Grace Harris off Lily Bassingthwaighte, a well-made 35 from just 16 balls, with 6 fours and a six, Harris departs! Laura Harris c Grace Harris b Lily Bassingthwaighte 35(16) [4s-6 6s-1]
Wicket, caught.
16.3
Lily Bassingthwaighte to Laura Harris, wide.
16.2
4
Lily Bassingthwaighte to Laura Harris, FOUR, Laura Harris pulls it away to deep square leg, and it races across the rope for four!
16.1
Lily Bassingthwaighte to Heather Kn

# **inspect the HTML structure**

In [131]:

# Inspect the HTML structure around the commentary section

heading = soup.find(
    string=lambda text: text and "Ball-by-ball commentary" in text
)

if heading:
    element = heading.parent

    print("COMMENTARY HEADING ANCESTORS\n")

    for level in range(6):
        if element is None:
            break

        print(f"\nLEVEL {level}")
        print("Tag:", element.name)
        print("Attributes:", element.attrs)
        print("Text:", element.get_text(" ", strip=True)[:500])

        element = element.parent


# Inspect HTML elements containing a delivery number

print("\n\nDELIVERY ELEMENTS\n")

for node in soup.find_all(string=lambda text: text and text.strip() == "16.4")[:5]:
    element = node.parent

    print("\nDelivery candidate:")
    print(element.prettify()[:1500])


COMMENTARY HEADING ANCESTORS


LEVEL 0
Tag: h2
Attributes: {'id': 'h-comm'}
Text: Ball-by-ball commentary

LEVEL 1
Tag: section
Attributes: {'class': ['comm-sec'], 'aria-labelledby': 'h-comm'}
Text: Ball-by-ball commentary All Wickets 4s & 6s Overs 16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four! Nadine de Klerk comes in, a right-handed bat. 16.3 W Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harris!! What a stunning catch, and it's a big wicket too, Laura Harris is caught by Grace Harris off Lily Bassingthwaighte, a well-made 35 from just 16 balls, with 6 fours and a si

LEVEL 2
Tag: div
Attributes: {'class': ['m-main']}
Text: BRHW 178/5  ·  SYTW 180/4 Sydney Thunder Women won by 6 wkts Read the match report Sydney Thunder Women beat Brisbane Heat Women by 6 wickets Highlights Official videos, played by YouTube. Watching one means accepting the YouTube Terms of Ser

# **Find the individual delivery containers**

In [132]:

commentary = soup.select_one("section.comm-sec")

balls = commentary.select("span.ball-over")

print("Total ball-number elements:", len(balls))

for i, ball in enumerate(balls[:5]):
    print(f"\n{'='*60}")
    print("DELIVERY", i + 1)

    element = ball

    for level in range(5):
        if element is None:
            break

        print(
            f"Level {level}:",
            element.name,
            element.get("class"),
            "|",
            element.get_text(" ", strip=True)[:350]
        )

        element = element.parent


Total ball-number elements: 178

DELIVERY 1
Level 0: span ['ball-over'] | 16.4 4
Level 1: li ['ball', 'ball-b'] | 16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!
Level 2: ol ['comm'] | 16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four! Nadine de Klerk comes in, a right-handed bat. 16.3 W Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harris!! What a stunning catch, and it's a big wicket too, Laura Harris is caught
Level 3: section ['comm-sec'] | Ball-by-ball commentary All Wickets 4s & 6s Overs 16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four! Nadine de Klerk comes in, a right-handed bat. 16.3 W Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harr

## **DIAGNOSTIC CELL**

# **Extract all delivery records**

In [133]:
commentary = soup.select_one("section.comm-sec")
delivery_elements = commentary.select("li.ball:not(.ball-note)")

records = []

for delivery in delivery_elements:

    ball_element = delivery.select_one("span.ball-over")

    # Extract only the direct text node: 16.4
    ball_number = ball_element.contents[0].strip()

    # Extract indicator: 4, 6, W, etc.
    indicator_element = ball_element.select_one(".bb")
    indicator = indicator_element.get_text(strip=True) if indicator_element else None

    # Extract commentary paragraph only
    description = delivery.select_one(".ball-text p").get_text(" ", strip=True)

    records.append({
        "ball_number": ball_number,
        "indicator": indicator,
        "html_class": " ".join(delivery.get("class", [])),
        "commentary": description
    })

df = pd.DataFrame(records)

# Inspect the two deliveries from Lucy Hamilton at 14.3

for i, record in enumerate(records):
    if "Lucy Hamilton to Laura Harris, FOUR" in record["commentary"]:
        print("Record number:", i)
        print("Ball number in records:", repr(record["ball_number"]))

        print("Ball number in dataframe:",
              repr(df.loc[i, "ball_number"]))

        print("-------------------------")


print("CHECKING RECORDS BEFORE DATAFRAME")

for record in records:
    if "Lucy Hamilton" in record["commentary"]:
        print(record["ball_number"])
        print(record["commentary"])

print("Total deliveries:", len(df))

display(df.head(10))

Record number: 14
Ball number in records: '14.3'
Ball number in dataframe: '14.3'
-------------------------
CHECKING RECORDS BEFORE DATAFRAME
14.6
Lucy Hamilton to Laura Harris, 1 run, Laura Harris works it away to deep mid-wicket for a single.
14.5
Lucy Hamilton to Heather Knight, 1 run, Heather Knight clips it out towards deep mid-wicket for a single.
14.4
Lucy Hamilton to Laura Harris, 1 run, Laura Harris pushes it out towards sweeper cover for a single.
14.3
Lucy Hamilton to Laura Harris, FOUR, Laura Harris sweeps it away to backward square leg for four!
14.2
Lucy Hamilton to Laura Harris, SIX, Laura Harris connects cleanly and launches it all the way over deep mid-wicket for six!
14.1
Lucy Hamilton to Heather Knight, 3 runs, Heather Knight pings this into the deep backward point region, and they'll easily scamper back for three good runs!
12.6
Lucy Hamilton to Laura Harris, no run, A dot ball from Lucy Hamilton to Laura Harris, good defence there.
12.5
Lucy Hamilton to Laura Harri

,ball_number,indicator,html_class,commentary
0,16.4,4,ball ball-b,"Lily Bassingthwaighte to Nadine de Klerk, FOUR..."
1,16.3,W,ball ball-w,"Lily Bassingthwaighte to Laura Harris, out Cau..."
2,16.3,None,ball,"Lily Bassingthwaighte to Laura Harris, wide."
3,16.2,4,ball ball-b,"Lily Bassingthwaighte to Laura Harris, FOUR, L..."
4,16.1,None,ball,"Lily Bassingthwaighte to Heather Knight, 1 run..."
5,15.6,None,ball,"Sarah Glenn to Heather Knight, 1 run, Heather ..."
6,15.5,4,ball ball-b,"Sarah Glenn to Heather Knight, FOUR, Knight cl..."
7,15.4,4,ball ball-b,"Sarah Glenn to Heather Knight, FOUR, Knight dr..."
8,15.3,None,ball,"Sarah Glenn to Laura Harris, 1 run, Laura Harr..."
9,15.2,None,ball,"Sarah Glenn to Laura Harris, no run."


In [134]:
print("Original dataframe:")
print(df.loc[14, ["ball_number", "commentary"]].to_dict())

print("\nParsed dataframe:")
print(parsed.loc[14, ["ball_number", "commentary"]].to_dict())

print("\nAll 14.3 entries in original dataframe:")
print(
    df[df["ball_number"] == "14.3"][
        ["ball_number", "commentary"]
    ].to_string(index=False)
)

print("\nAll 14.3 entries in parsed dataframe:")
print(
    parsed[parsed["ball_number"] == "14.3"][
        ["ball_number", "commentary"]
    ].to_string(index=False)
)

Original dataframe:
{'ball_number': '14.3', 'commentary': 'Lucy Hamilton to Laura Harris, FOUR, Laura Harris sweeps it away to backward square leg for four!'}

Parsed dataframe:
{'ball_number': '14.3', 'commentary': '14.3 4 Lucy Hamilton to Laura Harris, FOUR, Laura Harris sweeps it away to backward square leg for four!'}

All 14.3 entries in original dataframe:
ball_number                                                                                                     commentary
       14.3              Lucy Hamilton to Laura Harris, FOUR, Laura Harris sweeps it away to backward square leg for four!
       14.3 Nadine de Klerk to Orla Prendergast, FOUR, Orla Prendergast gets this away fine down to the boundary for four!

All 14.3 entries in parsed dataframe:
ball_number                                                                                                            commentary
       14.3              14.3 4 Lucy Hamilton to Laura Harris, FOUR, Laura Harris sweeps it away 

# **Separate deliveries from commentary notes**

In [135]:

# Keep only actual delivery entries
delivery_records = []

for delivery in delivery_elements:

    classes = delivery.get("class", [])

    # Exclude player introductions, wicket notes and over updates
    if "ball-note" in classes:
        continue

    ball_element = delivery.select_one("span.ball-over")
    run_element = ball_element.select_one(".bb") if ball_element else None

    ball_number = ball_element.contents[0].strip() if ball_element else None
    indicator = run_element.get_text(strip=True) if run_element else None

    delivery_records.append({
        "ball_number": ball_number,
        "indicator": indicator,
        "html_class": " ".join(classes),
        "commentary": delivery.get_text(" ", strip=True)
    })

df = pd.DataFrame(delivery_records)

print("Total HTML elements:", len(delivery_elements))
print("Actual delivery records:", len(df))
print("\nClass distribution:")
print(df["html_class"].value_counts())

display(df.head(10))


Total HTML elements: 178
Actual delivery records: 178

Class distribution:
html_class
ball                  127
ball ball-b            33
ball ball-b ball-6     10
ball ball-w             8
Name: count, dtype: int64


,ball_number,indicator,html_class,commentary
0,16.4,4,ball ball-b,16.4 4 Lily Bassingthwaighte to Nadine de Kler...
1,16.3,W,ball ball-w,"16.3 W Lily Bassingthwaighte to Laura Harris, ..."
2,16.3,None,ball,"16.3 Lily Bassingthwaighte to Laura Harris, wide."
3,16.2,4,ball ball-b,"16.2 4 Lily Bassingthwaighte to Laura Harris, ..."
4,16.1,None,ball,"16.1 Lily Bassingthwaighte to Heather Knight, ..."
5,15.6,None,ball,"15.6 Sarah Glenn to Heather Knight, 1 run, Hea..."
6,15.5,4,ball ball-b,"15.5 4 Sarah Glenn to Heather Knight, FOUR, Kn..."
7,15.4,4,ball ball-b,"15.4 4 Sarah Glenn to Heather Knight, FOUR, Kn..."
8,15.3,None,ball,"15.3 Sarah Glenn to Laura Harris, 1 run, Laura..."
9,15.2,None,ball,"15.2 Sarah Glenn to Laura Harris, no run."


In [136]:
print("Records:")
print(repr(records[14]["ball_number"]))

print("\nDataFrame:")
print(repr(df.iloc[14]["ball_number"]))

print("\nDataFrame created from records again:")
test_df = pd.DataFrame(records)
print(repr(test_df.iloc[14]["ball_number"]))

print("\nAre the values different?")
print(df.iloc[14]["ball_number"] != test_df.iloc[14]["ball_number"])

Records:
'14.3'

DataFrame:
'14.3'

DataFrame created from records again:
'14.3'

Are the values different?
False


# **Extract batter, bowler and delivery outcome**

In [137]:
import re

def parse_delivery(row):

    text = row["commentary"]
    html_class = row["html_class"]

    # Remove ball number and indicator from commentary
    clean_text = re.sub(
        r"^\d+\.\d+\s*(?:[46W]\s*)?",
        "",
        text
    ).strip()

    # Extract bowler and batter
    match = re.match(r"(.+?) to (.+?),", clean_text)

    bowler = match.group(1).strip() if match else None
    batter = match.group(2).strip() if match else None

    lower = clean_text.lower()

    # Identify outcome
    if "ball-w" in html_class:
        outcome = "wicket"
    elif "ball-6" in html_class:
        outcome = "six"
    elif "ball-b" in html_class:
        outcome = "four"
    elif "wide" in lower:
        outcome = "wide"
    elif "no ball" in lower or "no-ball" in lower:
        outcome = "no_ball"
    elif "no run" in lower:
        outcome = "dot"
    elif "leg bye" in lower:
        outcome = "leg_bye"
    elif re.search(r"\bbye\b", lower):
        outcome = "bye"
    else:
        outcome = "runs"

    # Identify dismissal
    dismissal = None

    if outcome == "wicket":
        for dtype in [
            "run out", "caught", "bowled", "lbw",
            "stumped", "hit wicket", "retired hurt",
            "retired out", "obstructing the field"
        ]:
            if dtype in lower:
                dismissal = dtype
                break

    # Extract runs
    runs_match = re.search(r"\b(\d+)\s+runs?\b", lower)

    if outcome == "six":
        runs = 6
    elif outcome == "four":
        runs = 4
    elif outcome == "dot":
        runs = 0
    elif runs_match:
        runs = int(runs_match.group(1))
    else:
        runs = None

    return pd.Series({
        "bowler": bowler,
        "batter": batter,
        "outcome": outcome,
        "runs_from_commentary": runs,
        "dismissal_type": dismissal
    })


# Rebuild parsed dataframe
parsed = df.join(df.apply(parse_delivery, axis=1))

display(parsed.head(20))

,ball_number,indicator,html_class,commentary,bowler,batter,outcome,runs_from_commentary,dismissal_type
0,16.4,4,ball ball-b,16.4 4 Lily Bassingthwaighte to Nadine de Kler...,Lily Bassingthwaighte,Nadine de Klerk,four,4.0,None
1,16.3,W,ball ball-w,"16.3 W Lily Bassingthwaighte to Laura Harris, ...",Lily Bassingthwaighte,Laura Harris,wicket,NaN,caught
2,16.3,None,ball,"16.3 Lily Bassingthwaighte to Laura Harris, wide.",Lily Bassingthwaighte,Laura Harris,wide,NaN,None
3,16.2,4,ball ball-b,"16.2 4 Lily Bassingthwaighte to Laura Harris, ...",Lily Bassingthwaighte,Laura Harris,four,4.0,None
4,16.1,None,ball,"16.1 Lily Bassingthwaighte to Heather Knight, ...",Lily Bassingthwaighte,Heather Knight,runs,1.0,None
5,15.6,None,ball,"15.6 Sarah Glenn to Heather Knight, 1 run, Hea...",Sarah Glenn,Heather Knight,runs,1.0,None
6,15.5,4,ball ball-b,"15.5 4 Sarah Glenn to Heather Knight, FOUR, Kn...",Sarah Glenn,Heather Knight,four,4.0,None
7,15.4,4,ball ball-b,"15.4 4 Sarah Glenn to Heather Knight, FOUR, Kn...",Sarah Glenn,Heather Knight,four,4.0,None
8,15.3,None,ball,"15.3 Sarah Glenn to Laura Harris, 1 run, Laura...",Sarah Glenn,Laura Harris,runs,1.0,None
9,15.2,None,ball,"15.2 Sarah Glenn to Laura Harris, no run.",Sarah Glenn,Laura Harris,dot,0.0,None


## **DIAGNOSTIC**

In [138]:
print("Values in records:")

print([
    r["ball_number"]
    for r in records
    if "14.3" in r["commentary"]
])

print("\nValues in df:")

print(
    df.loc[
        df["commentary"].str.contains("14.3", na=False),
        "ball_number"
    ].tolist()
)

Values in records:
[]

Values in df:
['14.3', '14.3']


In [139]:
print([
    record["ball_number"]
    for record in records
    if "14.3" in record["commentary"]
])

[]


In [140]:
print(
    df.loc[
        df["commentary"].str.contains("14.3", na=False),
        "ball_number"
    ].tolist()
)

print("Data type:", df["ball_number"].dtype)

['14.3', '14.3']
Data type: object


# **Inspect one raw delivery**

In [141]:
commentary = soup.select_one("section.comm-sec")

for delivery in commentary.select("li.ball"):
    if delivery.select_one("span.ball-over"):
        print(delivery)
        break

<li class="ball ball-b" data-ts="1793518334461"><span class="ball-over">16.4<span class="bb bb-b bb-4" title="Four">4</span></span><div class="ball-text"><p>Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!</p></div></li>


In [142]:
print(df.columns.tolist())
print(parsed.columns.tolist())

['ball_number', 'indicator', 'html_class', 'commentary']
['ball_number', 'indicator', 'html_class', 'commentary', 'bowler', 'batter', 'outcome', 'runs_from_commentary', 'dismissal_type']


In [143]:
print("Commentary:", repr(df.loc[0, "commentary"]))
print("HTML class:", repr(df.loc[0, "html_class"]))

Commentary: "16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!"
HTML class: 'ball ball-b'


# **DIAGNOSTIC CELL**

In [144]:
# Find the delivery whose ball number is 14.3
for delivery in delivery_elements:

    ball_element = delivery.select_one("span.ball-over")

    if ball_element and "14.3" in ball_element.get_text():

        print("HTML:")
        print(ball_element)

        print("\nDirect text node:")
        print(repr(ball_element.contents[0]))

        print("\nAll text nodes:")
        print(ball_element.find_all(string=True))

        print("\nExtracted using contents[0]:")
        print(repr(ball_element.contents[0].strip()))

        break

HTML:
<span class="ball-over">14.3<span class="bb bb-b bb-4" title="Four">4</span></span>

Direct text node:
'14.3'

All text nodes:
['14.3', '4']

Extracted using contents[0]:
'14.3'


In [145]:
print(
    df.loc[
        df["commentary"].str.contains("14.3", na=False),
        ["ball_number", "indicator", "commentary"]
    ].to_string(index=False)
)

ball_number indicator                                                                                                            commentary
       14.3         4              14.3 4 Lucy Hamilton to Laura Harris, FOUR, Laura Harris sweeps it away to backward square leg for four!
       14.3         4 14.3 4 Nadine de Klerk to Orla Prendergast, FOUR, Orla Prendergast gets this away fine down to the boundary for four!


In [146]:
print(
    df[df["ball_number"] == "14.3"][
        ["ball_number", "indicator", "commentary"]
    ].to_string(index=False)
)

print("\nTotal deliveries:", len(df))

ball_number indicator                                                                                                            commentary
       14.3         4              14.3 4 Lucy Hamilton to Laura Harris, FOUR, Laura Harris sweeps it away to backward square leg for four!
       14.3         4 14.3 4 Nadine de Klerk to Orla Prendergast, FOUR, Orla Prendergast gets this away fine down to the boundary for four!

Total deliveries: 178


## **validate the extraction**

In [147]:
print("Duplicate ball numbers:")
print(df[df.duplicated("ball_number", keep=False)][
    ["ball_number", "indicator", "html_class", "commentary"]
].to_string(index=False))

print("\nIndicator distribution:")
print(df["indicator"].value_counts(dropna=False))

print("\nDelivery count:", len(df))

Duplicate ball numbers:
ball_number indicator         html_class                                                                                                                                                                                                                                                                                                                                       commentary
       16.4         4        ball ball-b                                                                                                                                                                                     16.4 4 Lily Bassingthwaighte to Nadine de Klerk, FOUR, Nadine de Klerk crashes it away to sweeper cover, and that's racing all the way to the rope for four!
       16.3         W        ball ball-w 16.3 W Lily Bassingthwaighte to Laura Harris, out Caught by Grace Harris!! What a stunning catch, and it's a big wicket too, Laura Harris is caught by Grace Harris off Lily Bassin

## **identify the innings**

In [148]:
print("Commentary section details:")

for section in soup.select("section.comm-sec"):
    print(
        "Class:", section.get("class"),
        "| Heading:", section.find_previous(["h2", "h3"]).get_text(strip=True)
        if section.find_previous(["h2", "h3"]) else "No heading",
        "| Deliveries:", len(section.select("li.ball:not(.ball-note)"))
    )

Commentary section details:
Class: ['comm-sec'] | Heading: Highlights | Deliveries: 178
